# 📄 Layout Detection (DocLayout-YOLO) + OCR (EasyOCR)

### Pipeline
```
PDF page → image → DocLayout-YOLO → bboxes + class labels
         → filter TEXT classes only → crop each bbox → EasyOCR → text
```

**DocLayout-YOLO class labels (DocStructBench model):**
| ID | Label | OCR? |
|----|-------|------|
| 0 | title | ✅ |
| 1 | plain text | ✅ |
| 2 | abandon (header/footer/noise) | ❌ |
| 3 | figure | ❌ |
| 4 | figure_caption | ✅ |
| 5 | table | ❌ |
| 6 | table_caption | ✅ |
| 7 | table_footnote | ✅ |
| 8 | isolate_formula | ❌ |
| 9 | formula_caption | ✅ |

> Enable GPU: **Runtime → Change runtime type → T4 GPU**

## Step 1 — Install Dependencies

In [ ]:
!pip install doclayout-yolo -q
!pip install huggingface_hub -q
!pip install easyocr -q
!pip install pdf2image -q
!apt-get install -y poppler-utils -q
print('✅ Done!')

## Step 2 — Imports

In [ ]:
import random
import numpy as np
from pathlib import Path
from collections import Counter

import torch
import cv2
from PIL import Image
from pdf2image import convert_from_path
from huggingface_hub import hf_hub_download
from doclayout_yolo import YOLOv10
import easyocr

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from google.colab import files

import warnings; warnings.filterwarnings('ignore')
print('✅ Imports OK')

## Step 3 — Load DocLayout-YOLO + EasyOCR

In [ ]:
# Download DocStructBench weights from HuggingFace (~60MB, cached after first run)
print('Downloading DocLayout-YOLO DocStructBench weights...')
model_path = hf_hub_download(
    repo_id='juliozhao/DocLayout-YOLO-DocStructBench',
    filename='doclayout_yolo_docstructbench_imgsz1024.pt'
)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
layout_model = YOLOv10(model_path)
print(f'✅ DocLayout-YOLO ready  (device: {DEVICE})')

# EasyOCR
ocr_reader = easyocr.Reader(['en'], gpu=(DEVICE == 'cuda'))
print('✅ EasyOCR ready')

## Step 4 — Upload PDFs

In [ ]:
uploaded = files.upload()
upload_dir = Path('/content/pdfs'); upload_dir.mkdir(exist_ok=True)

pdf_paths = []
for fname, content in uploaded.items():
    if not fname.lower().endswith('.pdf'):
        print(f'⚠️  Skipping {fname}'); continue
    p = upload_dir / fname
    p.write_bytes(content)
    pdf_paths.append(str(p))
    print(f'  ✅ {fname}  ({len(content)/1024:.1f} KB)')

print(f'\n{len(pdf_paths)} PDF(s) ready')

## Step 5 — Core Helpers

In [ ]:
DPI = 200   # Page rendering resolution. Increase to 300 for dense/small text.

# DocStructBench class ID → label name
ID_TO_NAME = {
    0: 'title',
    1: 'plain text',
    2: 'abandon',          # headers, footers, page numbers, watermarks → skip
    3: 'figure',
    4: 'figure_caption',
    5: 'table',
    6: 'table_caption',
    7: 'table_footnote',
    8: 'isolate_formula',
    9: 'formula_caption'
}

# Classes we want to send to EasyOCR
TEXT_CLASSES = {'title', 'plain text', 'figure_caption',
                'table_caption', 'table_footnote', 'formula_caption'}

# Bbox colours for visualisation
COLOURS = {
    'title':           'blue',
    'plain text':      'green',
    'abandon':         'gray',
    'figure':          'brown',
    'figure_caption':  'orange',
    'table':           'red',
    'table_caption':   'orange',
    'table_footnote':  'purple',
    'isolate_formula': 'magenta',
    'formula_caption': 'pink',
}


def run_layout_detection(page_img: Image.Image, conf: float = 0.2) -> list:
    """
    Run DocLayout-YOLO on a PIL page image.
    Returns list of dicts: {label, class_id, confidence, xyxy=[x0,y0,x1,y1]}
    Coordinates are in pixel space of the input image.
    """
    img_array = np.array(page_img)
    results = layout_model.predict(
        img_array,
        imgsz=1024,
        conf=conf,
        device=DEVICE,
        verbose=False
    )[0]

    detections = []
    boxes   = results.boxes.xyxy.cpu().numpy()    # [N, 4]  x0 y0 x1 y1  (pixels)
    classes = results.boxes.cls.cpu().numpy()     # [N]     class ids
    scores  = results.boxes.conf.cpu().numpy()    # [N]     confidences

    for box, cls_id, score in zip(boxes, classes, scores):
        cls_id = int(cls_id)
        detections.append({
            'label':      ID_TO_NAME.get(cls_id, f'class_{cls_id}'),
            'class_id':   cls_id,
            'confidence': float(score),
            'xyxy':       [int(v) for v in box]   # [x0, y0, x1, y1]
        })

    # Sort top → bottom by y0
    detections.sort(key=lambda d: d['xyxy'][1])
    return detections


def crop_detection(page_img: Image.Image, det: dict, padding: int = 4) -> Image.Image | None:
    """Crop a detection bbox from the page image with optional pixel padding."""
    x0, y0, x1, y1 = det['xyxy']
    W, H = page_img.size
    x0 = max(0, x0 - padding);  x1 = min(W, x1 + padding)
    y0 = max(0, y0 - padding);  y1 = min(H, y1 + padding)
    if x1 <= x0 or y1 <= y0:
        return None
    return page_img.crop((x0, y0, x1, y1))


def visualise_detections(page_img: Image.Image, detections: list,
                         title: str = '') -> None:
    """Draw all detected bboxes on the page image, colour-coded by class."""
    W, H = page_img.size
    fig, ax = plt.subplots(1, 1, figsize=(11, 15))
    ax.imshow(page_img)
    ax.set_title(title, fontsize=11)
    ax.axis('off')

    for det in detections:
        x0, y0, x1, y1 = det['xyxy']
        label  = det['label']
        color  = COLOURS.get(label, 'gray')
        conf   = det['confidence']
        rect = patches.Rectangle(
            (x0, y0), x1-x0, y1-y0,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x0, max(y0-4, 0), f'{label} ({conf:.2f})',
                fontsize=7, color=color,
                bbox=dict(facecolor='white', alpha=0.5, pad=1, linewidth=0),
                clip_on=True)

    handles = [patches.Patch(color=c, label=l) for l, c in COLOURS.items()]
    ax.legend(handles=handles, loc='upper right', fontsize=7, framealpha=0.85)
    plt.tight_layout()
    plt.show()


def ocr_detections(page_img: Image.Image, detections: list) -> list:
    """
    Run EasyOCR on every TEXT_CLASS detection.
    Returns list of (label, text) in top-to-bottom order.
    """
    lines = []
    text_dets = [d for d in detections if d['label'] in TEXT_CLASSES]
    for det in text_dets:
        crop = crop_detection(page_img, det)
        if crop is None:
            continue
        result = ocr_reader.readtext(np.array(crop), detail=0)
        text   = ' '.join(result).strip()
        if text:
            lines.append((det['label'], text))
    return lines


print('✅ Helpers defined')

## Step 6 — Process Each PDF: Layout Detection → Visualise → OCR

Target pages:
- `Researchpaper_KAI` → page **6**
- `apple 10k KAI` → page **38**
- `Laptopcatalogue_KAI` → page **5**
- `20_Medical_Reports_KAI` → random
- `20_Restaurant_Bills_KAI` → random

In [ ]:
# ── Target page map (stem → page number, 1-based) ─────────────
PAGE_MAP = {
    'researchpaper_kai':    6,
    'apple 10k kai':        38,
    'laptopcatalogue_kai':  5,
}  # anything not listed → random page


for pdf_path in pdf_paths:
    filename   = Path(pdf_path).name
    stem_lower = Path(pdf_path).stem.lower()

    print(f'\n{"="*65}')
    print(f'📄  PDF : {filename}')

    # ── A. Count pages ────────────────────────────────────────
    # Convert only page 1 first to get page count cheaply
    all_pages_info = convert_from_path(pdf_path, dpi=72)   # low DPI just for count
    total_pages = len(all_pages_info)
    del all_pages_info   # free memory
    print(f'    Total pages: {total_pages}')

    # ── B. Determine target page ──────────────────────────────
    target_pg = PAGE_MAP.get(stem_lower)
    if target_pg is None or target_pg > total_pages:
        target_pg = random.randint(1, total_pages)
        print(f'    Random page selected: {target_pg}')
    else:
        print(f'    Target page: {target_pg}')

    # ── C. Render target page at full DPI ─────────────────────
    print(f'    Rendering page {target_pg} at {DPI} DPI...')
    page_imgs = convert_from_path(
        pdf_path, dpi=DPI, first_page=target_pg, last_page=target_pg
    )
    page_img = page_imgs[0]
    print(f'    Image size: {page_img.size[0]}×{page_img.size[1]} px')

    # ── D. DocLayout-YOLO detection ───────────────────────────
    print(f'    Running DocLayout-YOLO...')
    detections = run_layout_detection(page_img, conf=0.2)
    print(f'    Detections: {len(detections)}')

    # Label breakdown
    label_counts = Counter(d['label'] for d in detections)
    print(f'    Label breakdown: {dict(label_counts)}')

    # ── E. Visualise ALL bboxes ───────────────────────────────
    print(f'    Visualising layout...')
    visualise_detections(
        page_img, detections,
        title=f'{filename}  |  Page {target_pg}/{total_pages}  — DocLayout-YOLO'
    )

    # ── F. EasyOCR on text-class regions ─────────────────────
    text_count = sum(1 for d in detections if d['label'] in TEXT_CLASSES)
    skip_count = len(detections) - text_count
    print(f'    Text regions to OCR : {text_count}')
    print(f'    Skipped (table/fig/abandon/formula): {skip_count}')
    print(f'    Running EasyOCR...\n')

    print(f'{"─"*65}')
    print(f'  EXTRACTED TEXT | {filename} | Page {target_pg}')
    print(f'{"─"*65}')

    ocr_lines = ocr_detections(page_img, detections)

    if not ocr_lines:
        print('  [No text detected — try lowering conf threshold or increasing DPI]')
    else:
        for label, text in ocr_lines:
            prefix = '### ' if label == 'title' else ''
            print(prefix + text)

    print(f'{"─"*65}')

print('\n✅ Done! Compare output against your PDF pages.')

## Step 7 — Re-run on Any Specific Page
Change `TARGET_PDF_IDX` and `TARGET_PAGE` as needed.

In [ ]:
TARGET_PDF_IDX = 0   # index into pdf_paths list
TARGET_PAGE    = 1   # 1-based page number
CONF_THRESHOLD = 0.2 # lower this (e.g. 0.1) to detect more regions

pdf2    = pdf_paths[TARGET_PDF_IDX]
fname2  = Path(pdf2).name

imgs2 = convert_from_path(pdf2, dpi=DPI, first_page=TARGET_PAGE, last_page=TARGET_PAGE)
img2  = imgs2[0]

dets2 = run_layout_detection(img2, conf=CONF_THRESHOLD)
print(f'Detections on page {TARGET_PAGE}: {Counter(d["label"] for d in dets2)}')

visualise_detections(img2, dets2,
    title=f'{fname2}  |  Page {TARGET_PAGE}  (conf≥{CONF_THRESHOLD})')

print(f'\n--- EXTRACTED TEXT | {fname2} | Page {TARGET_PAGE} ---\n')
for label, text in ocr_detections(img2, dets2):
    print(('### ' if label == 'title' else '') + text)
print(f'\n--- END ---')

## Step 8 — Tune Confidence Threshold

If too many or too few regions are detected, adjust `conf`:
- **Lower conf (e.g. 0.1)** → detects more regions, may include false positives  
- **Higher conf (e.g. 0.5)** → only high-confidence detections, may miss some text blocks

Use the visualisation to judge quality before running OCR.

In [ ]:
# Quick sweep to find best conf threshold for a given page
TEST_PDF  = pdf_paths[0]
TEST_PAGE = 1

imgs_t = convert_from_path(TEST_PDF, dpi=DPI, first_page=TEST_PAGE, last_page=TEST_PAGE)
img_t  = imgs_t[0]

print(f'Detections count vs confidence threshold  |  {Path(TEST_PDF).name} p.{TEST_PAGE}\n')
print(f'{"Conf":>6}  {"Total":>6}  Label breakdown')
print('─' * 55)
for conf in [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]:
    dets = run_layout_detection(img_t, conf=conf)
    counts = dict(Counter(d['label'] for d in dets))
    print(f'{conf:>6.2f}  {len(dets):>6}  {counts}')